# P06 — Perturbation Sensitivity (Geneformer)

## Purpose
This notebook reproduces the complete in silico gene deletion and masking experiments from the original Geneformer research notebooks.

We perform three complementary perturbation analyses:
1. **Synthetic cell perturbation** — Delete each test gene from 50 synthetic cells, measure CLS embedding shift (cosine distance)
2. **Mask vs. delete** — Compare masking-in-place vs. delete-and-reindex to rule out positional artefacts
3. **Real PBMC validation** — Validate synthetic results using actual PBMC3k scRNA-seq data

### Output Files
- `perturbation_sensitivity.csv` — gene-level perturbation metrics (synthetic)
- `mask_perturbation_comparison.csv` — mask vs. delete comparison
- `perturbation_real_pbmc.csv` — real cell perturbation results
- `perturbation_synthetic_vs_real.csv` — merged synthetic/real comparison

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm.auto import tqdm
import scipy.sparse as sp

# ── Configuration ─────────────────────────────────────────────────────────
MODEL_PATH       = Path('Geneformer')
MODEL_DIR        = MODEL_PATH / 'Geneformer-V2-104M'
TOKEN_DICT_PATH  = MODEL_PATH / 'geneformer' / 'token_dictionary_gc104M.pkl'
GENE_NAME_PATH   = MODEL_PATH / 'geneformer' / 'gene_name_id_dict_gc104M.pkl'
MEDIAN_DICT_PATH = MODEL_PATH / 'geneformer' / 'gene_median_dictionary_gc104M.pkl'
DATA_DIR         = Path('data')

N_CELLS          = 50
CELL_GENES       = 1200
GENE_NOISE_STD   = 0.4
MODEL_INPUT_SIZE = 4096
FORWARD_BATCH    = 8
N_GENES_PER_GROUP = 50
ANOMALY_THRESHOLD = 3.0

DEVICE = 'cpu'
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
print(f'Device: {DEVICE}')

Device: mps


## 1. Load Model and Dictionaries

In [2]:
from transformers import BertModel

print('Loading Geneformer V2-104M...')
model = BertModel.from_pretrained(str(MODEL_DIR))
model.eval()
model.to(DEVICE)
print(f'Model on {DEVICE}  |  layers={model.config.num_hidden_layers}  '
      f'hidden={model.config.hidden_size}')

with open(TOKEN_DICT_PATH, 'rb') as f:
    token_dict = pickle.load(f)
with open(GENE_NAME_PATH, 'rb') as f:
    gene_name_dict = pickle.load(f)
with open(MEDIAN_DICT_PATH, 'rb') as f:
    median_expr_dict = pickle.load(f)

PAD_ID  = int(token_dict['<pad>'])
CLS_ID  = int(token_dict['<cls>'])
EOS_ID  = int(token_dict['<eos>'])
MASK_ID = int(token_dict['<mask>'])
print(f'Vocab: {len(token_dict)} tokens  PAD={PAD_ID} CLS={CLS_ID} EOS={EOS_ID} MASK={MASK_ID}')

Some weights of BertModel were not initialized from the model checkpoint at Geneformer/Geneformer-V2-104M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading Geneformer V2-104M...


Model on mps  |  layers=12  hidden=768
Vocab: 20275 tokens  PAD=0 CLS=2 EOS=3 MASK=1


## 2. Select Glitch and Control Gene Sets

In [3]:
df = pd.read_csv(DATA_DIR / 'gene_embedding_geometry.csv')
df = df[df['ensembl_id'].isin(token_dict)].copy()
df['token_id_val'] = df['ensembl_id'].map(token_dict)
df['median_expr'] = df['ensembl_id'].map(median_expr_dict)
df = df[df['median_expr'].notna() & (df['median_expr'] > 0)].copy()
df['log_median_expr'] = np.log1p(df['median_expr'])

print(f'Genes with vocab + expression data: {len(df)}')
print(f'Genes above anomaly threshold ({ANOMALY_THRESHOLD}): '
      f'{(df["anomaly_score_with_isolation"] > ANOMALY_THRESHOLD).sum()}')

# ── Glitch set
glitch_df = (
    df[df['anomaly_score_with_isolation'] > ANOMALY_THRESHOLD]
    .nlargest(N_GENES_PER_GROUP, 'anomaly_score_with_isolation')
    .copy()
)
glitch_df['group'] = 'glitch'

# ── Control set: expression-matched via quantile binning
median_anomaly = df['anomaly_score_with_isolation'].median()
candidate_ctrl = df[
    (df['anomaly_score_with_isolation'] < ANOMALY_THRESHOLD) &
    (~df['ensembl_id'].isin(glitch_df['ensembl_id']))
].copy()

glitch_df['expr_bin'] = pd.qcut(glitch_df['log_median_expr'], q=5, labels=False, duplicates='drop')
candidate_ctrl['expr_bin'] = pd.cut(
    candidate_ctrl['log_median_expr'],
    bins=pd.qcut(glitch_df['log_median_expr'], q=5, duplicates='drop', retbins=True)[1],
    labels=False, include_lowest=True
)

ctrl_parts = []
for bin_id, grp in glitch_df.groupby('expr_bin', observed=True):
    n_needed = len(grp)
    pool = candidate_ctrl[candidate_ctrl['expr_bin'] == bin_id].copy()
    pool['dist_to_median'] = (pool['anomaly_score_with_isolation'] - median_anomaly).abs()
    pool = pool.nsmallest(n_needed, 'dist_to_median')
    ctrl_parts.append(pool)

ctrl_df = pd.concat(ctrl_parts).head(N_GENES_PER_GROUP).copy()
ctrl_df['group'] = 'control'

test_genes = pd.concat([glitch_df, ctrl_df], ignore_index=True)

print(f'\nGlitch genes selected:  {len(glitch_df)}')
print(f'Control genes selected: {len(ctrl_df)}')
print(f'\nExpression matching:')
print(f'  Glitch  log_median_expr: {glitch_df["log_median_expr"].median():.3f}')
print(f'  Control log_median_expr: {ctrl_df["log_median_expr"].median():.3f}')

Genes with vocab + expression data: 20271
Genes above anomaly threshold (3.0): 410

Glitch genes selected:  50
Control genes selected: 45

Expression matching:
  Glitch  log_median_expr: 3.259
  Control log_median_expr: 3.189


## 3. Construct Synthetic Test Cells

In [4]:
# Gene pool: genes with both expression and token data
all_genes_pool = [(eid, val) for eid, val in median_expr_dict.items() if eid in token_dict]
all_gene_ids   = np.array([g[0] for g in all_genes_pool])
all_gene_exprs = np.log1p(np.array([g[1] for g in all_genes_pool]))
all_token_ids  = np.array([int(token_dict[g]) for g in all_gene_ids])  # int() cast!

print(f'Gene pool: {len(all_gene_ids)} genes with expression + token data')

rng = np.random.default_rng(42)
cells_input_ids_list = []
cells_attn_masks_list = []
cells_gene_sets = []

for cell_idx in range(N_CELLS):
    noise = rng.normal(0, GENE_NOISE_STD, len(all_gene_exprs))
    simulated_expr = all_gene_exprs + noise

    # Rank descending: highest expression first (matching gf_03 pattern)
    order = np.argsort(simulated_expr)[::-1]
    top_order = order[:CELL_GENES]

    gene_ids_cell  = all_gene_ids[top_order]
    token_ids_cell = all_token_ids[top_order]

    # Build sequence: [CLS] + genes + [EOS] + padding
    seq = np.concatenate([
        [CLS_ID],
        token_ids_cell,
        [EOS_ID],
    ])
    seq_len = len(seq)
    pad_len = MODEL_INPUT_SIZE - seq_len
    seq = np.concatenate([seq, np.full(pad_len, PAD_ID, dtype=np.int64)])

    attn = np.concatenate([
        np.ones(seq_len, dtype=np.int64),
        np.zeros(pad_len, dtype=np.int64),
    ])

    cells_input_ids_list.append(seq.astype(np.int64))
    cells_attn_masks_list.append(attn)
    cells_gene_sets.append(set(gene_ids_cell))

# Stack into tensors
cells_input_ids = torch.tensor(np.stack(cells_input_ids_list), dtype=torch.long)
cells_attn_masks_t = torch.tensor(np.stack(cells_attn_masks_list), dtype=torch.long)

print(f'Synthetic cells: {cells_input_ids.shape}  (cells x seq_len={MODEL_INPUT_SIZE})')
print(f'Sequence structure: [CLS] + {CELL_GENES} genes + [EOS] + {MODEL_INPUT_SIZE - CELL_GENES - 2} PADs')

# Coverage check for test genes
coverage = []
for _, row in test_genes.iterrows():
    n_present = sum(row['ensembl_id'] in s for s in cells_gene_sets)
    coverage.append(n_present / N_CELLS)
test_genes['cell_coverage'] = coverage
low_cov = test_genes[test_genes['cell_coverage'] < 0.5]
print(f'\nTest gene coverage: mean={np.mean(coverage):.2f}')
if len(low_cov):
    print(f'  WARNING: {len(low_cov)} genes in <50% of cells')

Gene pool: 20271 genes with expression + token data
Synthetic cells: torch.Size([50, 4096])  (cells x seq_len=4096)
Sequence structure: [CLS] + 1200 genes + [EOS] + 2894 PADs

Test gene coverage: mean=0.99


## 4. Baseline Embeddings and Perturbation Functions

In [5]:
@torch.no_grad()
def get_cls_embeddings(input_ids, attention_mask, batch_size=FORWARD_BATCH):
    """Extract CLS embeddings (position 0) from batched forward passes."""
    all_embs = []
    n = input_ids.shape[0]
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        ids  = input_ids[start:end].to(DEVICE)
        mask = attention_mask[start:end].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        cls_emb = out.last_hidden_state[:, 0, :].cpu()
        all_embs.append(cls_emb)
    return torch.cat(all_embs, dim=0)

print('Computing baseline CLS embeddings...')
baseline_embs = get_cls_embeddings(cells_input_ids, cells_attn_masks_t)
print(f'Baseline embeddings: {baseline_embs.shape}')

cells_np = cells_input_ids.numpy()
masks_np = cells_attn_masks_t.numpy()

def delete_gene_token(seq_np, token_id, pad_id):
    """Delete first occurrence of token_id and append pad_id."""
    idx = np.where(seq_np == token_id)[0]
    if len(idx) == 0:
        return seq_np.copy()
    new_seq = np.delete(seq_np, idx[0])
    new_seq = np.append(new_seq, pad_id)
    assert len(new_seq) == len(seq_np), f'Sequence length mismatch after deletion'
    return new_seq.astype(np.int64)

def mask_gene_token(seq_np, token_id, mask_id):
    """Replace first occurrence of token_id with mask_id in place."""
    idx = np.where(seq_np == token_id)[0]
    if len(idx) == 0:
        return seq_np.copy()
    new_seq = seq_np.copy()
    new_seq[idx[0]] = mask_id
    return new_seq.astype(np.int64)

MIN_CELLS = max(10, int(0.4 * N_CELLS))
print(f'Minimum cells per gene for analysis: {MIN_CELLS}')

Computing baseline CLS embeddings...


Baseline embeddings: torch.Size([50, 768])
Minimum cells per gene for analysis: 20


## 5. Part I — Synthetic Cell Perturbation Loop

For each test gene, delete it from all synthetic cells that contain it and measure the CLS embedding shift (cosine distance). This produces `perturbation_sensitivity.csv`.

**Expected runtime:** ~1–2 hours on Apple Silicon MPS.

In [6]:
print('\n' + '='*80)
print('PART I: SYNTHETIC CELL PERTURBATION')
print('='*80)

results = []

for gene_idx, (_, gene_row) in enumerate(tqdm(test_genes.iterrows(), total=len(test_genes), desc='Genes')):
    ensembl_id = gene_row['ensembl_id']
    gene_name = gene_row.get('gene', ensembl_id)
    group = gene_row['group']
    anomaly_score = gene_row['anomaly_score_with_isolation']
    token_id = int(gene_row['token_id_val'])
    median_expr = gene_row['median_expr']
    log_median_expr = gene_row['log_median_expr']
    
    # Find cells containing this gene
    cell_indices = [i for i, genes in enumerate(cells_gene_sets) if ensembl_id in genes]
    
    if len(cell_indices) < MIN_CELLS:
        continue
    
    # Delete gene token from all containing cells
    perturbed_seqs = []
    perturbed_masks = []
    for cell_idx in cell_indices:
        seq_copy = cells_np[cell_idx].copy()
        mask_copy = masks_np[cell_idx].copy()
        
        # Delete the gene token
        seq_perturbed = delete_gene_token(seq_copy, token_id, int(PAD_ID))
        
        # Update attention mask
        mask_perturbed = np.where(seq_perturbed == int(PAD_ID), 0, 1)
        
        perturbed_seqs.append(seq_perturbed)
        perturbed_masks.append(mask_perturbed)
    
    perturbed_seqs = torch.tensor(perturbed_seqs, dtype=torch.long)
    perturbed_masks = torch.tensor(perturbed_masks, dtype=torch.long)
    
    # Compute perturbed embeddings
    perturbed_embs = get_cls_embeddings(perturbed_seqs, perturbed_masks)
    
    # Compute cosine distances (1 - cosine_similarity)
    baseline_subset = baseline_embs[cell_indices]
    shifts = 1.0 - F.cosine_similarity(baseline_subset, perturbed_embs)
    shifts_np = shifts.numpy()
    
    # Record statistics
    results.append({
        'ensembl_id': ensembl_id,
        'gene': gene_name,
        'group': group,
        'anomaly_score': anomaly_score,
        'norm': gene_row.get('norm', np.nan),
        'isolation_score': gene_row.get('isolation_score', np.nan),
        'log_median_expr': log_median_expr,
        'n_cells': len(cell_indices),
        'mean_shift': float(np.mean(shifts_np)),
        'std_shift': float(np.std(shifts_np)),
        'median_shift': float(np.median(shifts_np)),
        'max_shift': float(np.max(shifts_np)),
        'cv_shift': float(np.std(shifts_np) / (np.mean(shifts_np) + 1e-8)) if np.mean(shifts_np) > 0 else 0.0
    })
    
    if (gene_idx + 1) % 10 == 0:
        print(f'  {gene_idx + 1}/{len(test_genes)} genes processed')

print(f'\nSynthetic perturbation complete: {len(results)} genes')


PART I: SYNTHETIC CELL PERTURBATION


Genes:   0%|          | 0/95 [00:00<?, ?it/s]

  10/95 genes processed


  20/95 genes processed


  30/95 genes processed


...

  60/95 genes processed


  70/95 genes processed


  80/95 genes processed


  90/95 genes processed



Synthetic perturbation complete: 95 genes


In [7]:
results_df = pd.DataFrame(results)
out_path = DATA_DIR / 'perturbation_sensitivity.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(results_df)} rows)')
print(f'  Glitch: {(results_df["group"]=="glitch").sum()}  '
      f'Control: {(results_df["group"]=="control").sum()}')
print(f'\nColumns: {list(results_df.columns)}')
print(f'\nFirst 5 rows:')
print(results_df.head())

Saved: data/perturbation_sensitivity.csv  (95 rows)
  Glitch: 50  Control: 45

Columns: ['ensembl_id', 'gene', 'group', 'anomaly_score', 'norm', 'isolation_score', 'log_median_expr', 'n_cells', 'mean_shift', 'std_shift', 'median_shift', 'max_shift', 'cv_shift']

First 5 rows:
        ensembl_id    gene   group  anomaly_score      norm  isolation_score  \
0  ENSG00000078328  RBFOX1  glitch       6.041130  2.083070         0.794868   
1  ENSG00000185774  KCNIP4  glitch       5.429368  1.964655         0.791874   
2  ENSG00000187391   MAGI2  glitch       5.331979  1.956460         0.761051   
3  ENSG00000147488    ST18  glitch       4.984153  1.902997         0.708462   
4  ENSG00000166710     B2M  glitch       4.934250  1.848038         0.787045   

   log_median_expr  n_cells  mean_shift  std_shift  median_shift  max_shift  \
0         3.498622       50    0.000172   0.000020      0.000173   0.000230   
1         3.699347       50    0.000146   0.000022      0.000145   0.000195   
2    

## 6. Part II — Mask-in-Place vs Delete-and-Reindex

Compare two perturbation operators to rule out positional artefacts:
- **Delete-and-reindex**: Remove the gene token and shift all downstream tokens left
- **Mask-in-place**: Replace the gene token with [MASK] at the same position

This produces `mask_perturbation_comparison.csv`.

In [8]:
print('\n' + '='*80)
print('PART II: MASK VS. DELETE COMPARISON')
print('='*80)

mask_results = []

for gene_idx, (_, gene_row) in enumerate(tqdm(test_genes.iterrows(), total=len(test_genes), desc='Genes')):
    ensembl_id = gene_row['ensembl_id']
    gene_name = gene_row.get('gene', ensembl_id)
    group = gene_row['group']
    anomaly_score = gene_row['anomaly_score_with_isolation']
    token_id = int(gene_row['token_id_val'])
    
    # Find cells containing this gene
    cell_indices = [i for i, genes in enumerate(cells_gene_sets) if ensembl_id in genes]
    
    if len(cell_indices) < MIN_CELLS:
        continue
    
    # Operator 1: Delete-and-reindex
    delete_seqs = []
    delete_masks = []
    for cell_idx in cell_indices:
        seq_copy = cells_np[cell_idx].copy()
        seq_delete = delete_gene_token(seq_copy, token_id, int(PAD_ID))
        mask_delete = np.where(seq_delete == int(PAD_ID), 0, 1)
        delete_seqs.append(seq_delete)
        delete_masks.append(mask_delete)
    
    delete_seqs = torch.tensor(delete_seqs, dtype=torch.long)
    delete_masks = torch.tensor(delete_masks, dtype=torch.long)
    delete_embs = get_cls_embeddings(delete_seqs, delete_masks)
    delete_shifts = (1.0 - F.cosine_similarity(baseline_embs[cell_indices], delete_embs)).numpy()
    
    # Operator 2: Mask-in-place
    mask_seqs = []
    mask_masks = []
    for cell_idx in cell_indices:
        seq_copy = cells_np[cell_idx].copy()
        seq_mask = mask_gene_token(seq_copy, token_id, int(MASK_ID))
        mask_mask = np.where(seq_mask == int(PAD_ID), 0, 1)
        mask_seqs.append(seq_mask)
        mask_masks.append(mask_mask)
    
    mask_seqs = torch.tensor(mask_seqs, dtype=torch.long)
    mask_masks = torch.tensor(mask_masks, dtype=torch.long)
    mask_embs = get_cls_embeddings(mask_seqs, mask_masks)
    mask_shifts = (1.0 - F.cosine_similarity(baseline_embs[cell_indices], mask_embs)).numpy()
    
    # Detect glitch (large divergence between mask and delete)
    shift_diff = np.abs(delete_shifts - mask_shifts)
    is_glitch = bool(np.mean(shift_diff) > np.std(shift_diff) + 1.0)
    
    mask_results.append({
        'gene': gene_name,
        'ensembl_id': ensembl_id,
        'group': group,
        'anomaly_score': anomaly_score,
        'n_cells': len(cell_indices),
        'delete_shift': float(np.mean(delete_shifts)),
        'mask_shift': float(np.mean(mask_shifts)),
        'delete_shift_median': float(np.median(delete_shifts)),
        'mask_shift_median': float(np.median(mask_shifts)),
        'is_glitch': is_glitch
    })
    
    if (gene_idx + 1) % 10 == 0:
        print(f'  {gene_idx + 1}/{len(test_genes)} genes processed')

print(f'\nMask vs. delete comparison complete: {len(mask_results)} genes')


PART II: MASK VS. DELETE COMPARISON


Genes:   0%|          | 0/95 [00:00<?, ?it/s]

  10/95 genes processed


  20/95 genes processed


  30/95 genes processed


...

  60/95 genes processed


  70/95 genes processed


  80/95 genes processed


  90/95 genes processed



Mask vs. delete comparison complete: 95 genes


In [9]:
mask_df = pd.DataFrame(mask_results)
mask_path = DATA_DIR / 'mask_perturbation_comparison.csv'
mask_df.to_csv(mask_path, index=False)
print(f'Saved: {mask_path}  ({len(mask_df)} rows)')
print(f'  Glitches detected: {mask_df["is_glitch"].sum()}')
print(f'\nColumns: {list(mask_df.columns)}')
print(f'\nFirst 5 rows:')
print(mask_df.head())

Saved: data/mask_perturbation_comparison.csv  (95 rows)
  Glitches detected: 0

Columns: ['gene', 'ensembl_id', 'group', 'anomaly_score', 'n_cells', 'delete_shift', 'mask_shift', 'delete_shift_median', 'mask_shift_median', 'is_glitch']

First 5 rows:
     gene       ensembl_id   group  anomaly_score  n_cells  delete_shift  \
0  RBFOX1  ENSG00000078328  glitch       6.041130       50      0.000172   
1  KCNIP4  ENSG00000185774  glitch       5.429368       50      0.000146   
2   MAGI2  ENSG00000187391  glitch       5.331979       50      0.000211   
3    ST18  ENSG00000147488  glitch       4.984153       50      0.000066   
4     B2M  ENSG00000166710  glitch       4.934250       50      0.000216   

   mask_shift  delete_shift_median  mask_shift_median  is_glitch  
0    0.000631             0.000173           0.000602      False  
1    0.000670             0.000145           0.000612      False  
2    0.000916             0.000207           0.000902      False  
3    0.000789           

## 7. Part III — Real PBMC Validation

Validate synthetic-cell perturbation results using real PBMC3k scRNA-seq data. This produces `perturbation_real_pbmc.csv` and `perturbation_synthetic_vs_real.csv`.

In [10]:
print('\n' + '='*80)
print('PART III: REAL PBMC VALIDATION')
print('='*80)

try:
    import scanpy as sc
    print('\nLoading PBMC3k dataset...')

    # ── Use RAW PBMC3k data (13,714 genes) ────────────────────────────
    # The processed version has only ~1,838 highly-variable genes, which
    # drastically reduces overlap with our 95 test genes (from ~25 to 1).
    # Using raw counts preserves the full gene universe, matching the
    # approach in the original gf_04a notebook.
    adata_proc = sc.datasets.pbmc3k_processed()
    adata_raw  = sc.datasets.pbmc3k()
    sc.pp.filter_cells(adata_raw, min_genes=200)
    sc.pp.filter_genes(adata_raw, min_cells=3)

    common_bc = adata_raw.obs_names.intersection(adata_proc.obs_names)
    adata = adata_raw[common_bc].copy()
    adata.obs['cell_type'] = adata_proc.obs.loc[common_bc, 'louvain'].values
    del adata_proc, adata_raw

    print(f'PBMC3k shape: {adata.shape}  ({adata.n_obs} cells, {adata.n_vars} genes)')
    
    # Get gene names and expression matrix
    gene_names = np.array(adata.var_names)
    X = adata.X.toarray() if sp.issparse(adata.X) else adata.X
    
    # Map gene symbols to Ensembl IDs
    # gene_name_dict is inverted (Ensembl -> name), so reverse it
    ensembl_to_name = {v: k for k, v in gene_name_dict.items()}
    name_to_ensembl = {v: k for k, v in ensembl_to_name.items()}
    
    # Find genes that are in both PBMC and our vocabulary
    pbmc_genes_in_vocab = []
    pbmc_gene_indices = []
    pbmc_ensembl_ids = []
    pbmc_token_ids = []
    pbmc_medians = []
    
    for gene_idx, gene_name in enumerate(gene_names):
        if gene_name in name_to_ensembl:
            ensembl_id = name_to_ensembl[gene_name]
            if ensembl_id in token_dict:
                pbmc_genes_in_vocab.append(gene_name)
                pbmc_gene_indices.append(gene_idx)
                pbmc_ensembl_ids.append(ensembl_id)
                pbmc_token_ids.append(int(token_dict[ensembl_id]))
                pbmc_medians.append(median_expr_dict.get(ensembl_id, 0))
    
    print(f'PBMC genes in Geneformer vocab: {len(pbmc_genes_in_vocab)}')
    
    # Use subset of PBMC cells for efficiency
    n_pbmc_cells = min(100, adata.n_obs)
    rng_pbmc = np.random.default_rng(42)
    pbmc_cell_indices = rng_pbmc.choice(adata.n_obs, n_pbmc_cells, replace=False)
    X_subset = X[pbmc_cell_indices, :][:, pbmc_gene_indices]
    
    print(f'Using {n_pbmc_cells} PBMC cells for validation')
    
    # Tokenize PBMC cells (rank-value encoding)
    pbmc_cells_input_ids = []
    pbmc_cells_attn_masks = []
    pbmc_cells_gene_sets = []
    
    for cell_idx in range(n_pbmc_cells):
        expr = X_subset[cell_idx, :]
        ranked_idx = np.argsort(-expr)  # Descending by expression
        
        seq = [int(CLS_ID)]
        genes_in_cell = set()
        for rank_pos, gene_idx in enumerate(ranked_idx):
            if expr[gene_idx] > 0:  # Only include expressed genes
                token_id = pbmc_token_ids[gene_idx]
                seq.append(token_id)
                genes_in_cell.add(pbmc_ensembl_ids[gene_idx])
        
        seq.append(int(EOS_ID))
        pbmc_cells_gene_sets.append(genes_in_cell)
        
        # Pad to MODEL_INPUT_SIZE
        if len(seq) < MODEL_INPUT_SIZE:
            seq.extend([int(PAD_ID)] * (MODEL_INPUT_SIZE - len(seq)))
        else:
            seq = seq[:MODEL_INPUT_SIZE]
        
        pbmc_cells_input_ids.append(seq)
        attn_mask = [1 if s != int(PAD_ID) else 0 for s in seq]
        pbmc_cells_attn_masks.append(attn_mask)
    
    pbmc_cells_input_ids = torch.tensor(pbmc_cells_input_ids, dtype=torch.long)
    pbmc_cells_attn_masks = torch.tensor(pbmc_cells_attn_masks, dtype=torch.long)
    
    print(f'PBMC cells tokenized: {pbmc_cells_input_ids.shape}')
    
except Exception as e:
    print(f'Warning: Could not load real PBMC data: {e}')
    print('Skipping real PBMC validation')
    pbmc_cells_input_ids = None




PART III: REAL PBMC VALIDATION



Loading PBMC3k dataset...


PBMC3k shape: (2638, 13714)  (2638 cells, 13714 genes)
PBMC genes in Geneformer vocab: 11159
Using 100 PBMC cells for validation


PBMC cells tokenized: torch.Size([100, 4096])


In [11]:
if pbmc_cells_input_ids is not None:
    print('Computing baseline PBMC embeddings...')
    pbmc_baseline_embs = get_cls_embeddings(pbmc_cells_input_ids, pbmc_cells_attn_masks)
    pbmc_cells_np = pbmc_cells_input_ids.numpy()
    pbmc_masks_np = pbmc_cells_attn_masks.numpy()
    
    print('Running perturbation on real PBMC cells...')
    pbmc_results = []
    
    # Filter test_genes to those present in PBMC
    test_genes_in_pbmc = test_genes[test_genes['ensembl_id'].isin(pbmc_ensembl_ids)].copy()
    
    for gene_idx, (_, gene_row) in enumerate(tqdm(test_genes_in_pbmc.iterrows(), total=len(test_genes_in_pbmc), desc='Genes')):
        ensembl_id = gene_row['ensembl_id']
        gene_name = gene_row.get('gene', ensembl_id)
        group = gene_row['group']
        anomaly_score = gene_row['anomaly_score_with_isolation']
        
        # Find token ID in PBMC vocab
        if ensembl_id not in pbmc_ensembl_ids:
            continue
        pbmc_gene_pos = pbmc_ensembl_ids.index(ensembl_id)
        token_id = pbmc_token_ids[pbmc_gene_pos]
        
        # Find PBMC cells containing this gene
        cell_indices = [i for i, genes in enumerate(pbmc_cells_gene_sets) if ensembl_id in genes]
        
        if len(cell_indices) < max(5, int(0.1 * n_pbmc_cells)):
            continue
        
        # Delete gene token
        perturbed_seqs = []
        perturbed_masks = []
        for cell_idx in cell_indices:
            seq_copy = pbmc_cells_np[cell_idx].copy()
            seq_perturbed = delete_gene_token(seq_copy, token_id, int(PAD_ID))
            mask_perturbed = np.where(seq_perturbed == int(PAD_ID), 0, 1)
            perturbed_seqs.append(seq_perturbed)
            perturbed_masks.append(mask_perturbed)
        
        perturbed_seqs = torch.tensor(perturbed_seqs, dtype=torch.long)
        perturbed_masks = torch.tensor(perturbed_masks, dtype=torch.long)
        perturbed_embs = get_cls_embeddings(perturbed_seqs, perturbed_masks)
        
        shifts = (1.0 - F.cosine_similarity(pbmc_baseline_embs[cell_indices], perturbed_embs)).numpy()
        
        pbmc_results.append({
            'ensembl_id': ensembl_id,
            'gene': gene_name,
            'group': group,
            'anomaly_score': anomaly_score,
            'n_cells': len(cell_indices),
            'mean_shift': float(np.mean(shifts)),
            'median_shift': float(np.median(shifts)),
            'max_shift': float(np.max(shifts))
        })
        
        if (gene_idx + 1) % 10 == 0:
            print(f'  {gene_idx + 1}/{len(test_genes_in_pbmc)} genes processed')
    
    pbmc_results_df = pd.DataFrame(pbmc_results)
    pbmc_path = DATA_DIR / 'perturbation_real_pbmc.csv'
    pbmc_results_df.to_csv(pbmc_path, index=False)
    print(f'\nSaved: {pbmc_path}  ({len(pbmc_results_df)} rows)')
else:
    pbmc_results_df = pd.DataFrame()  # Empty if PBMC loading failed

Computing baseline PBMC embeddings...


Running perturbation on real PBMC cells...


Genes:   0%|          | 0/29 [00:00<?, ?it/s]

  10/29 genes processed


  20/29 genes processed



Saved: data/perturbation_real_pbmc.csv  (24 rows)


In [12]:
# Merge synthetic and real PBMC results
if len(pbmc_results_df) > 0:
    print('Merging synthetic and real PBMC results...')
    
    # Rename synthetic columns for clarity
    synthetic_cols = results_df[['ensembl_id', 'gene', 'group', 'anomaly_score', 'mean_shift', 'median_shift']].copy()
    synthetic_cols.columns = ['ensembl_id', 'gene', 'group', 'anomaly_score', 'synthetic_mean_shift', 'synthetic_median_shift']
    
    # Rename real columns
    real_cols = pbmc_results_df[['ensembl_id', 'mean_shift', 'median_shift']].copy()
    real_cols.columns = ['ensembl_id', 'pbmc_mean_shift', 'pbmc_median_shift']
    
    # Merge on ensembl_id
    merged = synthetic_cols.merge(real_cols, on='ensembl_id', how='inner')
    
    # NOTE: Per-gene Spearman correlation (synthetic vs real shift) is computed
    # downstream in NB04 Cell 6 across all genes, not per-row.
    
    merged_path = DATA_DIR / 'perturbation_synthetic_vs_real.csv'
    merged.to_csv(merged_path, index=False)
    print(f'Saved: {merged_path}  ({len(merged)} rows)')
    print(f'\nColumns: {list(merged.columns)}')
else:
    print('Skipping synthetic/real merge due to missing real PBMC data')

Merging synthetic and real PBMC results...
Saved: data/perturbation_synthetic_vs_real.csv  (24 rows)

Columns: ['ensembl_id', 'gene', 'group', 'anomaly_score', 'synthetic_mean_shift', 'synthetic_median_shift', 'pbmc_mean_shift', 'pbmc_median_shift']


## 8. Verification

In [13]:
print('\n' + '='*80)
print('SUMMARY')
print('='*80)

# Verify all output files
output_files = [
    ('perturbation_sensitivity.csv', results_df),
    ('mask_perturbation_comparison.csv', mask_df),
]

if len(pbmc_results_df) > 0:
    output_files.append(('perturbation_real_pbmc.csv', pbmc_results_df))
    
if 'merged' in locals():
    output_files.append(('perturbation_synthetic_vs_real.csv', merged))

print(f'\nOutput files generated:')
for fname, df in output_files:
    fpath = DATA_DIR / fname
    print(f'  {fpath}')
    print(f'    Shape: {df.shape}')
    print(f'    Columns: {list(df.columns)}')
    print()

print('\nPerturbation sensitivity statistics:')
print(f'  Total genes analyzed: {len(results_df)}')
print(f'  Glitch genes: {(results_df["group"]=="glitch").sum()}')
print(f'  Control genes: {(results_df["group"]=="control").sum()}')
print(f'\n  Mean embedding shift:')
print(f'    Glitch  median: {results_df[results_df["group"]=="glitch"]["mean_shift"].median():.6f}')
print(f'    Control median: {results_df[results_df["group"]=="control"]["mean_shift"].median():.6f}')
print(f'\n  Median embedding shift:')
print(f'    Glitch  median: {results_df[results_df["group"]=="glitch"]["median_shift"].median():.6f}')
print(f'    Control median: {results_df[results_df["group"]=="control"]["median_shift"].median():.6f}')

print(f'\nMask vs. delete comparison statistics:')
print(f'  Total genes analyzed: {len(mask_df)}')
print(f'  Glitches detected: {mask_df["is_glitch"].sum()}')
print(f'  Correlation (mean shift): {mask_df[["delete_shift", "mask_shift"]].corr().iloc[0, 1]:.4f}')

print(f'\nAnalysis complete!')

# ── Sanity-check assertions ──────────────────────────────────────────────
assert 0 < len(results_df) <= 95, f"Expected 1-95 genes, got {len(results_df)}"
assert set(results_df['group'].unique()) == {'glitch', 'control'}, \
    f"Unexpected groups: {results_df['group'].unique()}"
assert (results_df['mean_shift'] >= 0).all(), "Negative mean shifts detected"
assert (results_df['median_shift'] >= 0).all(), "Negative median shifts detected"
if 'merged' in locals():
    assert len(merged) > 0, "Synthetic-vs-real merge produced no rows"
print('\nSanity checks passed.')


SUMMARY

Output files generated:
  data/perturbation_sensitivity.csv
    Shape: (95, 13)
    Columns: ['ensembl_id', 'gene', 'group', 'anomaly_score', 'norm', 'isolation_score', 'log_median_expr', 'n_cells', 'mean_shift', 'std_shift', 'median_shift', 'max_shift', 'cv_shift']

  data/mask_perturbation_comparison.csv
    Shape: (95, 10)
    Columns: ['gene', 'ensembl_id', 'group', 'anomaly_score', 'n_cells', 'delete_shift', 'mask_shift', 'delete_shift_median', 'mask_shift_median', 'is_glitch']

  data/perturbation_real_pbmc.csv
    Shape: (24, 8)
    Columns: ['ensembl_id', 'gene', 'group', 'anomaly_score', 'n_cells', 'mean_shift', 'median_shift', 'max_shift']

  data/perturbation_synthetic_vs_real.csv
    Shape: (24, 8)
    Columns: ['ensembl_id', 'gene', 'group', 'anomaly_score', 'synthetic_mean_shift', 'synthetic_median_shift', 'pbmc_mean_shift', 'pbmc_median_shift']


Perturbation sensitivity statistics:
  Total genes analyzed: 95
  Glitch genes: 50
  Control genes: 45

  Mean embed